In [1]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

TABLE_DIR = BASE_DIR / "notebooks" / "outputs" / "tables"

files = [
    "batter_high_pressure_stats.csv",
    "bowler_high_pressure_stats.csv",
    "batter_high_pressure_shortlist.csv",
    "bowler_high_pressure_shortlist.csv",
    "final_batter_rankings.csv",
    "final_bowler_rankings.csv"
]

for filename in files:
    path = TABLE_DIR / filename

    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{filename}")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
    else:
        print(f"\nMISSING: {filename}")



MISSING: batter_high_pressure_stats.csv

bowler_high_pressure_stats.csv
Shape: (536, 7)
Columns: ['bowler_id', 'bowler_name', 'high_pressure_wickets', 'legal_balls', 'pressure_deliveries', 'matches_played', 'wickets_per_24_balls']

MISSING: batter_high_pressure_shortlist.csv

bowler_high_pressure_shortlist.csv
Shape: (350, 8)
Columns: ['provisional_rank', 'bowler_id', 'bowler_name', 'high_pressure_wickets', 'legal_balls', 'pressure_deliveries', 'matches_played', 'wickets_per_24_balls']

final_batter_rankings.csv
Shape: (436, 18)
Columns: ['provisional_rank', 'striker_id', 'striker_name', 'high_pressure_runs', 'balls_faced', 'fours', 'sixes', 'pressure_deliveries', 'matches_played', 'strike_rate', 'boundary_runs', 'boundary_percentage', 'runs_percentile', 'strike_rate_percentile', 'boundary_percentile', 'sample_reliability', 'batter_score', 'final_rank']

final_bowler_rankings.csv
Shape: (350, 13)
Columns: ['provisional_rank', 'bowler_id', 'bowler_name', 'high_pressure_wickets', 'legal

In [2]:

# Load final ranking datasets

batter = pd.read_csv(TABLE_DIR / "final_batter_rankings.csv")
bowler = pd.read_csv(TABLE_DIR / "final_bowler_rankings.csv")

# Inspect batter scoring components
print("=" * 60)
print("BATTER SCORING COMPONENTS")
print("=" * 60)

batter_components = [
    "runs_percentile",
    "strike_rate_percentile",
    "boundary_percentile",
    "sample_reliability",
    "batter_score",
    "final_rank"
]

print(batter[batter_components].describe().round(4))

# Inspect bowler scoring components
print("\n" + "=" * 60)
print("BOWLER SCORING COMPONENTS")
print("=" * 60)

bowler_components = [
    "wickets_percentile",
    "wicket_rate_percentile",
    "sample_sufficiency",
    "bowler_score",
    "final_rank"
]

print(bowler[bowler_components].describe().round(4))

# Inspect the leading players
print("\nTOP 5 BATTERS")
print(
    batter.sort_values("final_rank")
    [["final_rank", "striker_name", "batter_score"]]
    .head(5).to_string(index=False)
)

print("\nTOP 5 BOWLERS")
print(
    bowler.sort_values("final_rank")
    [["final_rank", "bowler_name", "bowler_score"]]
    .head(5).to_string(index=False)
)


BATTER SCORING COMPONENTS
       runs_percentile  strike_rate_percentile  boundary_percentile  \
count         436.0000                436.0000             436.0000   
mean            0.5011                  0.5011               0.5011   
std             0.2890                  0.2890               0.2890   
min             0.0034                  0.0034               0.0161   
25%             0.2500                  0.2517               0.2523   
50%             0.5006                  0.5011               0.5011   
75%             0.7506                  0.7506               0.7506   
max             1.0000                  1.0000               1.0000   

       sample_reliability  batter_score  final_rank  
count            436.0000      436.0000    436.0000  
mean               0.6702       52.6499    218.5000  
std                0.3144       24.9461    126.0066  
min                0.2000        3.4817      1.0000  
25%                0.3500       32.5866    109.7500  
50%       

In [3]:

# ============================================================
# STEP 3: RANKING SENSITIVITY ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. DEFINE SCENARIOS
# ------------------------------------------------------------

batter_scenarios = {
    "Original": {
        "runs_percentile": 0.40,
        "strike_rate_percentile": 0.30,
        "boundary_percentile": 0.15,
        "sample_reliability": 0.15
    },
    "Balanced": {
        "runs_percentile": 0.30,
        "strike_rate_percentile": 0.25,
        "boundary_percentile": 0.20,
        "sample_reliability": 0.25
    },
    "Performance_Focused": {
        "runs_percentile": 0.50,
        "strike_rate_percentile": 0.25,
        "boundary_percentile": 0.15,
        "sample_reliability": 0.10
    }
}

bowler_scenarios = {
    "Original": {
        "wickets_percentile": 0.45,
        "wicket_rate_percentile": 0.30,
        "sample_sufficiency": 0.25
    },
    "Balanced": {
        "wickets_percentile": 0.40,
        "wicket_rate_percentile": 0.30,
        "sample_sufficiency": 0.30
    },
    "Performance_Focused": {
        "wickets_percentile": 0.55,
        "wicket_rate_percentile": 0.35,
        "sample_sufficiency": 0.10
    }
}


# ------------------------------------------------------------
# 2. GENERIC SENSITIVITY FUNCTION
# ------------------------------------------------------------

def run_sensitivity(data, scenarios, name_column, original_score_column):

    results = data.copy()

    # Check that every scoring component exists
    required_columns = sorted({
        column
        for weights in scenarios.values()
        for column in weights
    })

    missing = [
        column for column in required_columns
        if column not in results.columns
    ]

    if missing:
        raise ValueError(f"Missing scoring columns: {missing}")

    # Check original score reconstruction
    original_weights = scenarios["Original"]

    reconstructed = sum(
        results[column] * weight
        for column, weight in original_weights.items()
    ) * 100

    max_difference = (
        reconstructed - results[original_score_column]
    ).abs().max()

    print(
        f"\nMaximum original-score reconstruction difference: "
        f"{max_difference:.10f}"
    )

    if max_difference > 0.0001:
        raise ValueError(
            "Original score reconstruction does not match. "
            "Stop and investigate before comparing scenarios."
        )

    # Calculate scores and ranks for all scenarios
    for scenario_name, weights in scenarios.items():

        score_column = f"{scenario_name}_score"
        rank_column = f"{scenario_name}_rank"

        results[score_column] = sum(
            results[column] * weight
            for column, weight in weights.items()
        ) * 100

        results[rank_column] = (
            results[score_column]
            .rank(method="first", ascending=False)
            .astype(int)
        )

    return results


# ------------------------------------------------------------
# 3. RUN BATTER AND BOWLER SENSITIVITY
# ------------------------------------------------------------

batter_sensitivity = run_sensitivity(
    batter,
    batter_scenarios,
    "striker_name",
    "batter_score"
)

bowler_sensitivity = run_sensitivity(
    bowler,
    bowler_scenarios,
    "bowler_name",
    "bowler_score"
)


# ------------------------------------------------------------
# 4. COMPARE TOP 10 OVERLAP AND RANK CORRELATION
# ------------------------------------------------------------

def compare_scenarios(results, name_column, original_rank_column):

    summary = []

    for scenario in ["Balanced", "Performance_Focused"]:

        alternative_rank = f"{scenario}_rank"

        original_top10 = set(
            results.loc[
                results[original_rank_column] <= 10,
                name_column
            ]
        )

        alternative_top10 = set(
            results.loc[
                results[alternative_rank] <= 10,
                name_column
            ]
        )

        common_players = original_top10.intersection(
            alternative_top10
        )

        overlap = len(common_players)

        rank_correlation = results[
            original_rank_column
        ].corr(
            results[alternative_rank],
            method="spearman"
        )

        summary.append({
            "Scenario": scenario,
            "Original_Top10": len(original_top10),
            "Alternative_Top10": len(alternative_top10),
            "Common_Players": overlap,
            "Top10_Overlap_Percent": overlap * 10,
            "Spearman_Rank_Correlation": rank_correlation
        })

    return pd.DataFrame(summary)


batter_comparison = compare_scenarios(
    batter_sensitivity,
    "striker_name",
    "final_rank"
)

bowler_comparison = compare_scenarios(
    bowler_sensitivity,
    "bowler_name",
    "final_rank"
)


# ------------------------------------------------------------
# 5. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BATTER SENSITIVITY RESULTS")
print("=" * 65)

print(batter_comparison.round(4).to_string(index=False))

print("\n" + "=" * 65)
print("BOWLER SENSITIVITY RESULTS")
print("=" * 65)

print(bowler_comparison.round(4).to_string(index=False))


# ------------------------------------------------------------
# 6. DISPLAY TOP 10 UNDER EACH SCENARIO
# ------------------------------------------------------------

for scenario in ["Original", "Balanced", "Performance_Focused"]:

    print(f"\nTOP 10 BATTERS — {scenario.upper()}")

    print(
        batter_sensitivity
        .sort_values(f"{scenario}_rank")
        [[
            "striker_name",
            f"{scenario}_score",
            f"{scenario}_rank"
        ]]
        .head(10)
        .round(2)
        .to_string(index=False)
    )

    print(f"\nTOP 10 BOWLERS — {scenario.upper()}")

    print(
        bowler_sensitivity
        .sort_values(f"{scenario}_rank")
        [[
            "bowler_name",
            f"{scenario}_score",
            f"{scenario}_rank"
        ]]
        .head(10)
        .round(2)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 7. EXPORT VALIDATION RESULTS
# ------------------------------------------------------------

VALIDATION_DIR = (
    BASE_DIR / "notebooks" / "outputs" / "validation"
)

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

batter_sensitivity.to_csv(
    VALIDATION_DIR / "batter_sensitivity_results.csv",
    index=False
)

bowler_sensitivity.to_csv(
    VALIDATION_DIR / "bowler_sensitivity_results.csv",
    index=False
)

batter_comparison.to_csv(
    VALIDATION_DIR / "batter_scenario_comparison.csv",
    index=False
)

bowler_comparison.to_csv(
    VALIDATION_DIR / "bowler_scenario_comparison.csv",
    index=False
)

print("\n" + "=" * 65)
print("SENSITIVITY ANALYSIS COMPLETED")
print("=" * 65)

print("Validation folder:", VALIDATION_DIR)
print("Original rankings have NOT been modified.")



Maximum original-score reconstruction difference: 0.0000000000

Maximum original-score reconstruction difference: 0.0000000000

BATTER SENSITIVITY RESULTS
           Scenario  Original_Top10  Alternative_Top10  Common_Players  Top10_Overlap_Percent  Spearman_Rank_Correlation
           Balanced              10                 10               9                     90                     0.9971
Performance_Focused              10                 10               9                     90                     0.9990

BOWLER SENSITIVITY RESULTS
           Scenario  Original_Top10  Alternative_Top10  Common_Players  Top10_Overlap_Percent  Spearman_Rank_Correlation
           Balanced              10                 10              10                    100                     0.9992
Performance_Focused              10                 10              10                    100                     0.9873

TOP 10 BATTERS — ORIGINAL
    striker_name  Original_score  Original_rank
Suryakumar Yada

In [4]:

# ============================================================
# STEP 4: PLAYER RANKING STABILITY REPORT
# ============================================================

def create_stability_report(data, name_column):

    report = data[[
        name_column,
        "Original_rank",
        "Balanced_rank",
        "Performance_Focused_rank"
    ]].copy()

    rank_columns = [
        "Original_rank",
        "Balanced_rank",
        "Performance_Focused_rank"
    ]

    # Number of scenarios in which player appears in Top 10
    report["Top10_Appearances"] = (
        report[rank_columns] <= 10
    ).sum(axis=1)

    # Average rank across all scenarios
    report["Average_Rank"] = (
        report[rank_columns].mean(axis=1)
    )

    # Difference between best and worst rank
    report["Rank_Range"] = (
        report[rank_columns].max(axis=1)
        - report[rank_columns].min(axis=1)
    )

    # Sort by Top 10 appearances, then average rank
    report = report.sort_values(
        ["Top10_Appearances", "Average_Rank"],
        ascending=[False, True]
    ).reset_index(drop=True)

    return report


# Generate reports
batter_stability = create_stability_report(
    batter_sensitivity,
    "striker_name"
)

bowler_stability = create_stability_report(
    bowler_sensitivity,
    "bowler_name"
)


# Display most stable players
print("=" * 70)
print("MOST STABLE BATTERS")
print("=" * 70)

print(
    batter_stability.head(15)
    .round(2)
    .to_string(index=False)
)

print("\n" + "=" * 70)
print("MOST STABLE BOWLERS")
print("=" * 70)

print(
    bowler_stability.head(15)
    .round(2)
    .to_string(index=False)
)


# Export reports
batter_stability.to_csv(
    VALIDATION_DIR / "batter_ranking_stability.csv",
    index=False
)

bowler_stability.to_csv(
    VALIDATION_DIR / "bowler_ranking_stability.csv",
    index=False
)


# Summary
print("\n" + "=" * 70)
print("STABILITY REPORT EXPORTED")
print("=" * 70)

print("Batter report: batter_ranking_stability.csv")
print("Bowler report: bowler_ranking_stability.csv")


MOST STABLE BATTERS
    striker_name  Original_rank  Balanced_rank  Performance_Focused_rank  Top10_Appearances  Average_Rank  Rank_Range
Suryakumar Yadav              1              1                         1                  3          1.00           0
       Tim David              2              2                         2                  3          2.00           0
 Nicholas Pooran              3              3                         3                  3          3.00           0
   Andre Russell              4              4                         4                  3          4.00           0
    Shreyas Iyer              5              5                         5                  3          5.00           0
Heinrich Klaasen              6              8                         6                  3          6.67           2
     Jos Buttler              7              7                         7                  3          7.00           0
 Ashutosh Sharma              8     

In [6]:

# ============================================================
# CORRECTED STEP 5: SAMPLE-SIZE RELIABILITY TEST
# ============================================================

def sample_threshold_test(
    data,
    player_column,
    sample_column,
    score_column,
    thresholds,
    label
):

    # Correct baseline: use the actual original final_rank
    original_top10 = set(
        data.loc[data["final_rank"] <= 10, player_column]
    )

    results = []

    for minimum in thresholds:

        eligible = data[
            data[sample_column] >= minimum
        ].copy()

        # Rank by highest original composite score
        eligible = eligible.sort_values(
            [score_column, player_column],
            ascending=[False, True]
        )

        new_top10 = set(
            eligible.head(10)[player_column]
        )

        retained = original_top10.intersection(new_top10)
        dropped = original_top10 - new_top10
        new_entries = new_top10 - original_top10

        results.append({
            "Role": label,
            "Minimum_Sample": minimum,
            "Eligible_Players": len(eligible),
            "Top10_Overlap": len(retained),
            "Overlap_Percent": len(retained) * 10,
            "Dropped_From_Original": ", ".join(sorted(dropped)),
            "New_Top10_Entries": ", ".join(sorted(new_entries))
        })

    return pd.DataFrame(results)


# Batter thresholds
batter_threshold_results = sample_threshold_test(
    data=batter_sensitivity,
    player_column="striker_name",
    sample_column="balls_faced",
    score_column="batter_score",
    thresholds=[20, 30, 40, 50, 75, 100],
    label="Batter"
)


# Bowler thresholds
bowler_threshold_results = sample_threshold_test(
    data=bowler_sensitivity,
    player_column="bowler_name",
    sample_column="legal_balls",
    score_column="bowler_score",
    thresholds=[24, 36, 48, 60, 90, 120],
    label="Bowler"
)


# Display results
print("=" * 75)
print("CORRECTED BATTER SAMPLE-SIZE RESULTS")
print("=" * 75)

print(batter_threshold_results.to_string(index=False))

print("\n" + "=" * 75)
print("CORRECTED BOWLER SAMPLE-SIZE RESULTS")
print("=" * 75)

print(bowler_threshold_results.to_string(index=False))


# Export corrected results
batter_threshold_results.to_csv(
    VALIDATION_DIR / "batter_sample_threshold_tests.csv",
    index=False
)

bowler_threshold_results.to_csv(
    VALIDATION_DIR / "bowler_sample_threshold_tests.csv",
    index=False
)

print("\nCorrected sample-size results exported.")


CORRECTED BATTER SAMPLE-SIZE RESULTS
  Role  Minimum_Sample  Eligible_Players  Top10_Overlap  Overlap_Percent Dropped_From_Original New_Top10_Entries
Batter              20               436             10              100                                        
Batter              30               350             10              100                                        
Batter              40               305             10              100                                        
Batter              50               271             10              100                                        
Batter              75               208             10              100                                        
Batter             100               171             10              100                                        

CORRECTED BOWLER SAMPLE-SIZE RESULTS
  Role  Minimum_Sample  Eligible_Players  Top10_Overlap  Overlap_Percent Dropped_From_Original New_Top10_Entries
Bowler              2

In [7]:

# ============================================================
# STEP 6A: INSPECT PROCESSED DATA FILES
# ============================================================

PROCESSED_DIR = BASE_DIR / "data" / "processed"

print("PROCESSED DATA DIRECTORY")
print(PROCESSED_DIR)

print("\nAVAILABLE FILES")
for file in sorted(PROCESSED_DIR.glob("*")):
    print(file.name)

print("\nPRESSURE / MATCH-STATE FILE INSPECTION")

for file in PROCESSED_DIR.glob("*.csv"):

    try:
        df = pd.read_csv(file, nrows=5)

        relevant = [
            col for col in df.columns
            if any(term in col.lower() for term in [
                "pressure",
                "innings_progress",
                "wicket_pressure",
                "legal_balls_before",
                "wickets_before",
                "runs_before",
                "state_quality",
                "striker_id",
                "bowler_id"
            ])
        ]

        if relevant:
            print(f"\nFile: {file.name}")
            print("Relevant columns:", relevant)

    except Exception as error:
        print(f"Could not inspect {file.name}: {error}")


PROCESSED DATA DIRECTORY
c:\Users\Dell\Downloads\SupeRR_Selector_Season2\data\processed

AVAILABLE FILES
selected_ball_by_ball.csv
selected_competitions.csv
selected_matches.csv

PRESSURE / MATCH-STATE FILE INSPECTION

File: selected_ball_by_ball.csv
Relevant columns: ['striker_id', 'non_striker_id', 'bowler_id']


In [8]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = BASE_DIR / "data" / "processed"

matches = pd.read_csv(PROCESSED_DIR / "selected_matches.csv")
competitions = pd.read_csv(PROCESSED_DIR / "selected_competitions.csv")

print("=" * 75)
print("SELECTED MATCHES — ALL COLUMNS")
print("=" * 75)

for col in matches.columns:
    print(col)

print("\n" + "=" * 75)
print("POTENTIAL MATCH-DURATION / INNINGS-LENGTH COLUMNS")
print("=" * 75)

keywords = [
    "over", "innings", "toss", "result",
    "date", "competition", "match_type", "format"
]

candidate_cols = [
    col for col in matches.columns
    if any(word in col.lower() for word in keywords)
]

print(matches[candidate_cols].head(10).to_string(index=False))

print("\n" + "=" * 75)
print("COMPETITION COLUMNS")
print("=" * 75)

print(competitions.columns.tolist())
print(competitions.head(5).to_string(index=False))

SELECTED MATCHES — ALL COLUMNS
match_id
comp_id
match_number
match_date
stage
venue_id
venue_country
home_team_id
home_team
away_team_id
away_team
home_team_captain_id
away_team_captain_id
toss_win_team_id
toss_opted
man_of_the_match
first_inn_revised_overs
second_inn_revised_overs
revised_target
knock_out
win_team_id
result

POTENTIAL MATCH-DURATION / INNINGS-LENGTH COLUMNS
match_date  toss_win_team_id toss_opted  first_inn_revised_overs  second_inn_revised_overs                                result
2022-01-22              28.0   Fielding                      NaN                       NaN          West Indies Won by 9 Wickets
2022-01-23              28.0   Fielding                      NaN                       NaN                  England Won by 1 Run
2022-01-26              22.0   Fielding                      NaN                       NaN            West Indies Won by 20 runs
2022-01-29              28.0   Fielding                      NaN                       NaN                

In [9]:
# Check revised innings duration and target information

revision_cols = [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]

print("=" * 75)
print("MISSING VALUE SUMMARY")
print("=" * 75)

print(matches[revision_cols].isna().sum().to_string())

print("\n" + "=" * 75)
print("POPULATED VALUE COUNTS")
print("=" * 75)

for col in revision_cols:
    print(f"\n{col}")
    print("Non-missing matches:", matches[col].notna().sum())
    print("Distinct values:", matches[col].nunique())
    print(matches[col].value_counts(dropna=True).sort_index().to_string())

print("\n" + "=" * 75)
print("MATCHES WITH REVISION INFORMATION")
print("=" * 75)

revised_matches = matches[
    matches[revision_cols].notna().any(axis=1)
].copy()

print("Total matches with any revision information:",
      len(revised_matches))

display(
    revised_matches[
        [
            "match_id",
            "match_date",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "result"
        ]
    ].sort_values("match_date").head(30)
)

MISSING VALUE SUMMARY
first_inn_revised_overs     828
second_inn_revised_overs    828
revised_target              830

POPULATED VALUE COUNTS

first_inn_revised_overs
Non-missing matches: 25
Distinct values: 13
first_inn_revised_overs
30.0     2
42.0     1
48.0     1
54.0     4
60.0     1
66.0     3
72.0     1
78.0     1
84.0     3
90.0     2
102.0    2
108.0    2
114.0    2

second_inn_revised_overs
Non-missing matches: 25
Distinct values: 12
second_inn_revised_overs
30.0     2
42.0     2
48.0     3
54.0     2
60.0     2
66.0     1
72.0     1
78.0     1
84.0     1
90.0     5
102.0    2
114.0    3

revised_target
Non-missing matches: 23
Distinct values: 23
revised_target
33.0     1
56.0     1
69.0     1
78.0     1
91.0     1
94.0     1
96.0     1
103.0    1
104.0    1
115.0    1
116.0    1
125.0    1
126.0    1
130.0    1
136.0    1
143.0    1
147.0    1
151.0    1
152.0    1
168.0    1
171.0    1
203.0    1
213.0    1

MATCHES WITH REVISION INFORMATION
Total matches with any revision 

,match_id,match_date,first_inn_revised_overs,second_inn_revised_overs,revised_target,result
8,13612,2022-02-11,NaN,114.0,143.0,Australia Won by 20 runs (D/L method)
94,15410,2022-06-19,114.0,NaN,NaN,Match Abandoned
95,15546,2022-07-02,84.0,NaN,NaN,No Result
143,16753,2022-08-15,66.0,NaN,NaN,Afghanistan Won by 27 Runs
144,16754,2022-08-17,NaN,42.0,56.0,Ireland Won by 7 Wickets(D/L Method)
152,17395,2022-09-23,48.0,48.0,91.0,India Won by 6 Wickets
177,17765,2022-10-14,72.0,72.0,130.0,No Result
263,19959,2023-03-25,66.0,NaN,NaN,West Indies won by 3 wickets
272,21070,2023-03-27,NaN,48.0,104.0,Bangladesh Won by 22 Runs (DLS method)
273,21071,2023-03-29,102.0,102.0,203.0,Bangladesh Won by 77 Runs


In [12]:
ball_data = pd.read_csv(PROCESSED_DIR / "selected_ball_by_ball.csv")

print("Shape:", ball_data.shape)

print("\nColumns:")
print(ball_data.columns.tolist())

print("\nRelevant delivery columns:")
possible_cols = [
    col for col in ball_data.columns
    if any(word in col.lower() for word in [
        "match", "innings", "over", "ball",
        "extras", "delivery", "wicket"
    ])
]

print(ball_data[possible_cols].head(10).to_string(index=False))

print("\nData types:")
print(ball_data[possible_cols].dtypes.to_string())

Shape: (194014, 33)

Columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']

Relevant delivery columns:
 match_id  innings  overid  over_ball_num  innings_ball_num  is_wicket extras_type  extras1_runs  extras_2_runs delivery_style over_or_around       delivery_ist_time
    13511        1       1              1                 1      False         NaN             6              0        Inswing           Over 2022-01-23 00:00:00 UTC
    13511        1       1              2                 2      False         NaN            

In [13]:
# Reconstruct legal-ball classification using the original methodology

legal_map = {
    "normal": True,
    "byes": True,
    "legbyes": True,
    "wide": False,
    "Overstep": False,
    "other": pd.NA,
    "Penalty": pd.NA
}

ball_data["delivery_type"] = (
    ball_data["extras_type"].fillna("normal")
)

ball_data["is_legal_ball"] = (
    ball_data["delivery_type"]
    .map(legal_map)
    .astype("boolean")
)

# Aggregate observed deliveries by match and innings
observed = (
    ball_data
    .groupby(["match_id", "innings"])
    .agg(
        observed_legal_balls=(
            "is_legal_ball",
            lambda x: x.eq(True).sum()
        ),
        unknown_delivery_types=(
            "is_legal_ball",
            lambda x: x.isna().sum()
        ),
        total_deliveries=("match_id", "size")
    )
    .reset_index()
)

# Attach revised match information
revision_audit = observed.merge(
    matches[
        [
            "match_id",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "result"
        ]
    ],
    on="match_id",
    how="left"
)

# Select the relevant revised duration for each innings
revision_audit["expected_legal_balls"] = (
    revision_audit["first_inn_revised_overs"]
    .where(
        revision_audit["innings"] == 1,
        revision_audit["second_inn_revised_overs"]
    )
)

revision_audit = revision_audit[
    revision_audit["expected_legal_balls"].notna()
].copy()

revision_audit["difference"] = (
    revision_audit["observed_legal_balls"]
    - revision_audit["expected_legal_balls"]
)

revision_audit["comparison"] = "Below revised duration"
revision_audit.loc[
    revision_audit["difference"] == 0, "comparison"
] = "Exactly matches"
revision_audit.loc[
    revision_audit["difference"] > 0, "comparison"
] = "Above revised duration"

print("Revised innings with ball-by-ball data:", len(revision_audit))

print("\nComparison summary:")
print(revision_audit["comparison"].value_counts().to_string())

print("\nUnknown delivery classifications:",
      revision_audit["unknown_delivery_types"].sum())

display(
    revision_audit[
        [
            "match_id",
            "innings",
            "expected_legal_balls",
            "observed_legal_balls",
            "difference",
            "unknown_delivery_types",
            "comparison",
            "result"
        ]
    ].sort_values(["match_id", "innings"])
)

Revised innings with ball-by-ball data: 48

Comparison summary:
comparison
Exactly matches           28
Below revised duration    20

Unknown delivery classifications: 4


,match_id,innings,expected_legal_balls,observed_legal_balls,difference,unknown_delivery_types,comparison,result
17,13612,2,114.0,114,0.0,0,Exactly matches,Australia Won by 20 runs (D/L method)
182,15410,1,114.0,21,-93.0,0,Below revised duration,Match Abandoned
189,15546,1,84.0,78,-6.0,1,Below revised duration,No Result
284,16753,1,66.0,66,0.0,0,Exactly matches,Afghanistan Won by 27 Runs
287,16754,2,42.0,40,-2.0,0,Below revised duration,Ireland Won by 7 Wickets(D/L Method)
302,17395,1,48.0,48,0.0,0,Exactly matches,India Won by 6 Wickets
303,17395,2,48.0,44,-4.0,0,Below revised duration,India Won by 6 Wickets
352,17765,1,72.0,72,0.0,0,Exactly matches,No Result
353,17765,2,72.0,23,-49.0,0,Below revised duration,No Result
376,19959,1,66.0,66,0.0,0,Exactly matches,West Indies won by 3 wickets


In [14]:
import numpy as np

# 1. Reconstruct delivery-level pressure state
pressure_test = ball_data.copy()

pressure_test = pressure_test[
    pressure_test["innings"].isin([1, 2])
].copy()

pressure_test = pressure_test.sort_values(
    ["match_id", "innings", "innings_ball_num"]
).reset_index(drop=True)

# Numeric indicators
pressure_test["legal_numeric"] = (
    pressure_test["is_legal_ball"]
    .eq(True).fillna(False).astype(int)
)

pressure_test["unknown_numeric"] = (
    pressure_test["is_legal_ball"].isna().astype(int)
)

pressure_test["wicket_numeric"] = (
    pressure_test["is_wicket"].astype(int)
)

group_cols = ["match_id", "innings"]

# State BEFORE the current delivery
pressure_test["legal_balls_before"] = (
    pressure_test.groupby(group_cols)["legal_numeric"].cumsum()
    - pressure_test["legal_numeric"]
)

pressure_test["unknown_before"] = (
    pressure_test.groupby(group_cols)["unknown_numeric"].cumsum()
    - pressure_test["unknown_numeric"]
)

pressure_test["wickets_before"] = (
    pressure_test.groupby(group_cols)["wicket_numeric"].cumsum()
    - pressure_test["wicket_numeric"]
)

pressure_test["pressure_eligible"] = (
    pressure_test["unknown_before"] == 0
)

# 2. Attach innings-specific revised duration
pressure_test = pressure_test.merge(
    matches[
        [
            "match_id",
            "first_inn_revised_overs",
            "second_inn_revised_overs"
        ]
    ],
    on="match_id",
    how="left"
)

pressure_test["revised_limit"] = (
    pressure_test["first_inn_revised_overs"]
    .where(
        pressure_test["innings"] == 1,
        pressure_test["second_inn_revised_overs"]
    )
)

# Use 120 balls where no revised duration is recorded
pressure_test["innings_limit"] = (
    pressure_test["revised_limit"].fillna(120)
)

# 3. Calculate original and alternative pressure scores
pressure_test["innings_progress_original"] = (
    pressure_test["legal_balls_before"] / 120
).clip(0, 1)

pressure_test["innings_progress_adjusted"] = (
    pressure_test["legal_balls_before"]
    / pressure_test["innings_limit"]
).clip(0, 1)

pressure_test["wicket_pressure"] = (
    pressure_test["wickets_before"] / 10
).clip(0, 1)

pressure_test["pressure_original"] = 100 * (
    0.60 * pressure_test["innings_progress_original"]
    + 0.40 * pressure_test["wicket_pressure"]
)

pressure_test["pressure_adjusted"] = 100 * (
    0.60 * pressure_test["innings_progress_adjusted"]
    + 0.40 * pressure_test["wicket_pressure"]
)

# 4. Compare only eligible deliveries
eligible_test = pressure_test[
    pressure_test["pressure_eligible"]
].copy()

eligible_test["score_difference"] = (
    eligible_test["pressure_adjusted"]
    - eligible_test["pressure_original"]
)

eligible_test["original_high_pressure"] = (
    eligible_test["pressure_original"] >= 50
)

eligible_test["adjusted_high_pressure"] = (
    eligible_test["pressure_adjusted"] >= 50
)

print("=" * 75)
print("PRESSURE DENOMINATOR SENSITIVITY")
print("=" * 75)

print("Total regulation deliveries:", len(pressure_test))
print("Eligible deliveries:", len(eligible_test))

print(
    "Deliveries in innings with revised duration:",
    (eligible_test["revised_limit"].notna()).sum()
)

print("\nOriginal pressure score:")
print(eligible_test["pressure_original"].describe().round(3))

print("\nAdjusted pressure score:")
print(eligible_test["pressure_adjusted"].describe().round(3))

print("\nScore difference:")
print(eligible_test["score_difference"].describe().round(3))

print("\nHigh-pressure classification:")
print(
    "Original high-pressure deliveries:",
    eligible_test["original_high_pressure"].sum()
)
print(
    "Adjusted high-pressure deliveries:",
    eligible_test["adjusted_high_pressure"].sum()
)
print(
    "Changed classification:",
    (
        eligible_test["original_high_pressure"]
        != eligible_test["adjusted_high_pressure"]
    ).sum()
)

PRESSURE DENOMINATOR SENSITIVITY
Total regulation deliveries: 193988
Eligible deliveries: 184726
Deliveries in innings with revised duration: 3135

Original pressure score:
count    184726.000
mean         38.410
std          24.268
min           0.000
25%          17.500
50%          37.000
75%          57.500
max          95.500
Name: pressure_original, dtype: float64

Adjusted pressure score:
count    184726.000
mean         38.557
std          24.294
min           0.000
25%          18.000
50%          37.000
75%          58.000
max          95.500
Name: pressure_adjusted, dtype: float64

Score difference:
count    184726.000
mean          0.147
std           1.573
min           0.000
25%           0.000
50%           0.000
75%           0.000
max          43.500
Name: score_difference, dtype: float64

High-pressure classification:
Original high-pressure deliveries: 62219
Adjusted high-pressure deliveries: 62769
Changed classification: 550


In [15]:
# Compare original and adjusted pressure models at leaderboard level

# Keep only deliveries eligible under the original state-quality rule
ranking_test = eligible_test.copy()

# Create two independent high-pressure datasets
original_hp = ranking_test[
    ranking_test["pressure_original"] >= 50
].copy()

adjusted_hp = ranking_test[
    ranking_test["pressure_adjusted"] >= 50
].copy()


# ---------------------------------------------------------
# BATTER RANKING FUNCTION
# ---------------------------------------------------------

def calculate_batter_ranking(data):

    data = data.copy()

    data["batter_runs"] = data["runs"]

    data["batter_ball_faced"] = (
        data["delivery_type"] != "wide"
    ).astype(int)

    data["four"] = (data["runs"] == 4).astype(int)
    data["six"] = (data["runs"] == 6).astype(int)

    stats = (
        data.groupby(["striker_id", "striker_name"])
        .agg(
            high_pressure_runs=("batter_runs", "sum"),
            balls_faced=("batter_ball_faced", "sum"),
            fours=("four", "sum"),
            sixes=("six", "sum"),
            pressure_deliveries=("match_id", "size"),
            matches_played=("match_id", "nunique")
        )
        .reset_index()
    )

    stats = stats[stats["balls_faced"] >= 20].copy()

    stats["strike_rate"] = (
        stats["high_pressure_runs"]
        / stats["balls_faced"] * 100
    )

    stats["boundary_runs"] = (
        stats["fours"] * 4 + stats["sixes"] * 6
    )

    stats["boundary_percentage"] = (
        stats["boundary_runs"]
        / stats["high_pressure_runs"].replace(0, np.nan)
        * 100
    ).fillna(0)

    stats["runs_percentile"] = (
        stats["high_pressure_runs"].rank(pct=True)
    )

    stats["strike_rate_percentile"] = (
        stats["strike_rate"].rank(pct=True)
    )

    stats["boundary_percentile"] = (
        stats["boundary_percentage"].rank(pct=True)
    )

    stats["sample_reliability"] = (
        stats["balls_faced"] / 100
    ).clip(upper=1)

    stats["batter_score"] = 100 * (
        0.40 * stats["runs_percentile"]
        + 0.30 * stats["strike_rate_percentile"]
        + 0.15 * stats["boundary_percentile"]
        + 0.15 * stats["sample_reliability"]
    )

    return stats.sort_values(
        "batter_score", ascending=False
    ).reset_index(drop=True)


# ---------------------------------------------------------
# BOWLER RANKING FUNCTION
# ---------------------------------------------------------

def calculate_bowler_ranking(data):

    data = data.copy()

    credited_types = [
        "Bowled",
        "Caught",
        "LBW",
        "Stump",
        "Hit Wicket"
    ]

    data["credited_wicket"] = (
        data["dismissal_type"].isin(credited_types).astype(int)
    )

    data["legal_ball"] = (
        data["is_legal_ball"].eq(True).fillna(False).astype(int)
    )

    stats = (
        data.groupby(["bowler_id", "bowler_name"])
        .agg(
            high_pressure_wickets=("credited_wicket", "sum"),
            legal_balls=("legal_ball", "sum"),
            pressure_deliveries=("match_id", "size"),
            matches_played=("match_id", "nunique")
        )
        .reset_index()
    )

    stats = stats[stats["legal_balls"] >= 24].copy()

    stats["wickets_per_24_balls"] = (
        stats["high_pressure_wickets"]
        / stats["legal_balls"] * 24
    )

    stats["wickets_percentile"] = (
        stats["high_pressure_wickets"].rank(pct=True)
    )

    stats["wicket_rate_percentile"] = (
        stats["wickets_per_24_balls"].rank(pct=True)
    )

    stats["sample_sufficiency"] = (
        stats["legal_balls"] / 120
    ).clip(upper=1)

    stats["bowler_score"] = 100 * (
        0.45 * stats["wickets_percentile"]
        + 0.30 * stats["wicket_rate_percentile"]
        + 0.25 * stats["sample_sufficiency"]
    )

    return stats.sort_values(
        "bowler_score", ascending=False
    ).reset_index(drop=True)


# Calculate alternative leaderboards
adjusted_batters = calculate_batter_ranking(adjusted_hp)
adjusted_bowlers = calculate_bowler_ranking(adjusted_hp)

# Original Top 10 from saved final rankings
original_batter_top10 = set(
    batter.loc[batter["final_rank"] <= 10, "striker_name"]
)

original_bowler_top10 = set(
    bowler.loc[bowler["final_rank"] <= 10, "bowler_name"]
)

# Adjusted Top 10
adjusted_batter_top10 = set(
    adjusted_batters.head(10)["striker_name"]
)

adjusted_bowler_top10 = set(
    adjusted_bowlers.head(10)["bowler_name"]
)

print("=" * 75)
print("ADJUSTED PRESSURE MODEL — LEADERBOARD VALIDATION")
print("=" * 75)

print("\nBATTER RESULTS")
print("Original Top 10:", len(original_batter_top10))
print("Adjusted Top 10:", len(adjusted_batter_top10))
print("Overlap:", len(original_batter_top10 & adjusted_batter_top10), "/ 10")
print("Dropped:", sorted(original_batter_top10 - adjusted_batter_top10))
print("New entries:", sorted(adjusted_batter_top10 - original_batter_top10))

print("\nAdjusted Batter Top 10:")
display(
    adjusted_batters.head(10)[
        ["striker_name", "batter_score", "high_pressure_runs",
         "balls_faced", "strike_rate"]
    ]
)

print("\nBOWLER RESULTS")
print("Original Top 10:", len(original_bowler_top10))
print("Adjusted Top 10:", len(adjusted_bowler_top10))
print("Overlap:", len(original_bowler_top10 & adjusted_bowler_top10), "/ 10")
print("Dropped:", sorted(original_bowler_top10 - adjusted_bowler_top10))
print("New entries:", sorted(adjusted_bowler_top10 - original_bowler_top10))

print("\nAdjusted Bowler Top 10:")
display(
    adjusted_bowlers.head(10)[
        ["bowler_name", "bowler_score", "high_pressure_wickets",
         "legal_balls", "wickets_per_24_balls"]
    ]
)

ADJUSTED PRESSURE MODEL — LEADERBOARD VALIDATION

BATTER RESULTS
Original Top 10: 10
Adjusted Top 10: 10
Overlap: 9 / 10
Dropped: ['Harry Brook']
New entries: ['Naman Dhir']

Adjusted Batter Top 10:


,striker_name,batter_score,high_pressure_runs,balls_faced,strike_rate
0,Suryakumar Yadav,97.734554,1116,575,194.086957
1,Tim David,95.720824,1447,814,177.764128
2,Nicholas Pooran,95.217391,1149,652,176.226994
3,Andre Russell,94.553776,1035,600,172.500000
4,Shreyas Iyer,93.466819,894,479,186.638831
5,Jos Buttler,93.272311,728,392,185.714286
6,Heinrich Klaasen,93.157895,1064,580,183.448276
7,Marcus Stoinis,92.231121,950,556,170.863309
8,Ashutosh Sharma,91.973684,522,293,178.156997
9,Naman Dhir,91.613272,451,243,185.596708



BOWLER RESULTS
Original Top 10: 10
Adjusted Top 10: 10
Overlap: 9 / 10
Dropped: ['Matheesha Pathirana']
New entries: ['Adam Zampa']

Adjusted Bowler Top 10:


,bowler_name,bowler_score,high_pressure_wickets,legal_balls,wickets_per_24_balls
0,Haris Rauf,93.678571,40,402,2.388060
1,Mitchell Starc,93.592857,28,237,2.835443
2,Kuldeep Yadav,92.864286,55,627,2.105263
3,Yuzvendra Chahal,92.800000,61,712,2.056180
4,Shaheen Afridi,92.392857,34,358,2.279330
5,Wanindu Hasaranga,92.157143,53,617,2.061588
6,Mark Adair,91.578571,31,328,2.268293
7,Josh Hazlewood,90.935714,41,487,2.020534
8,Andre Russell,90.721429,31,343,2.169096
9,Adam Zampa,90.528571,38,447,2.040268


In [16]:
# Recalculate the original fixed-120 model
baseline_batters = calculate_batter_ranking(original_hp)
baseline_bowlers = calculate_bowler_ranking(original_hp)

# Compare Top 10 membership
saved_batter_top10 = set(
    batter.loc[batter["final_rank"] <= 10, "striker_name"]
)

saved_bowler_top10 = set(
    bowler.loc[bowler["final_rank"] <= 10, "bowler_name"]
)

recalculated_batter_top10 = set(
    baseline_batters.head(10)["striker_name"]
)

recalculated_bowler_top10 = set(
    baseline_bowlers.head(10)["bowler_name"]
)

print("=" * 75)
print("ORIGINAL MODEL REPRODUCTION CHECK")
print("=" * 75)

print("\nBATTERS")
print("Top 10 overlap:",
      len(saved_batter_top10 & recalculated_batter_top10), "/ 10")
print("Missing from recalculated:",
      sorted(saved_batter_top10 - recalculated_batter_top10))
print("Unexpected entries:",
      sorted(recalculated_batter_top10 - saved_batter_top10))

print("\nBOWLERS")
print("Top 10 overlap:",
      len(saved_bowler_top10 & recalculated_bowler_top10), "/ 10")
print("Missing from recalculated:",
      sorted(saved_bowler_top10 - recalculated_bowler_top10))
print("Unexpected entries:",
      sorted(recalculated_bowler_top10 - saved_bowler_top10))

# Compare scores for players appearing in both saved and recalculated Top 10
batter_score_check = (
    batter.loc[batter["final_rank"] <= 10, ["striker_name", "batter_score"]]
    .merge(
        baseline_batters[["striker_name", "batter_score"]],
        on="striker_name",
        how="inner",
        suffixes=("_saved", "_recalculated")
    )
)

bowler_score_check = (
    bowler.loc[bowler["final_rank"] <= 10, ["bowler_name", "bowler_score"]]
    .merge(
        baseline_bowlers[["bowler_name", "bowler_score"]],
        on="bowler_name",
        how="inner",
        suffixes=("_saved", "_recalculated")
    )
)

print("\nMaximum absolute score difference")
print(
    "Batters:",
    (batter_score_check["batter_score_saved"]
     - batter_score_check["batter_score_recalculated"]).abs().max()
)
print(
    "Bowlers:",
    (bowler_score_check["bowler_score_saved"]
     - bowler_score_check["bowler_score_recalculated"]).abs().max()
)

ORIGINAL MODEL REPRODUCTION CHECK

BATTERS
Top 10 overlap: 10 / 10
Missing from recalculated: []
Unexpected entries: []

BOWLERS
Top 10 overlap: 10 / 10
Missing from recalculated: []
Unexpected entries: []

Maximum absolute score difference
Batters: 0.0
Bowlers: 0.0


In [17]:
# Audit match formats and revised-duration metadata

format_audit = matches.merge(
    competitions[["comp_id", "match_format"]],
    on="comp_id",
    how="left"
)

print("=" * 75)
print("MATCH FORMAT AUDIT")
print("=" * 75)

print("\nSelected match formats:")
display(
    format_audit["match_format"]
    .value_counts(dropna=False)
    .rename_axis("match_format")
    .reset_index(name="matches")
)

print("\nMatch formats among matches with revised-duration metadata:")

revised_matches = format_audit[
    format_audit[
        [
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ].notna().any(axis=1)
].copy()

display(
    revised_matches.groupby("match_format", dropna=False)
    .agg(
        matches_with_revised_metadata=("match_id", "nunique"),
        first_innings_limits=("first_inn_revised_overs", "count"),
        second_innings_limits=("second_inn_revised_overs", "count"),
        revised_targets=("revised_target", "count")
    )
    .reset_index()
)

print("\nRevised innings-duration values (recorded in balls):")
for col in ["first_inn_revised_overs", "second_inn_revised_overs"]:
    print(f"\n{col}")
    print(
        sorted(
            revised_matches[col].dropna().unique().tolist()
        )
    )

print("\nMissing match-format values:")
print(format_audit["match_format"].isna().sum())

MATCH FORMAT AUDIT

Selected match formats:


,match_format,matches
0,T20I,482
1,T20,371



Match formats among matches with revised-duration metadata:


,match_format,matches_with_revised_metadata,first_innings_limits,second_innings_limits,revised_targets
0,T20,5,3,5,5
1,T20I,29,22,20,18



Revised innings-duration values (recorded in balls):

first_inn_revised_overs
[30.0, 42.0, 48.0, 54.0, 60.0, 66.0, 72.0, 78.0, 84.0, 90.0, 102.0, 108.0, 114.0]

second_inn_revised_overs
[30.0, 42.0, 48.0, 54.0, 60.0, 66.0, 72.0, 78.0, 84.0, 90.0, 102.0, 114.0]

Missing match-format values:
0


In [18]:
# Identify matches responsible for pressure classification changes

changed = eligible_test[
    eligible_test["original_high_pressure"]
    != eligible_test["adjusted_high_pressure"]
].copy()

changed_match_audit = (
    changed.groupby("match_id")
    .agg(
        changed_deliveries=("match_id", "size"),
        original_high_pressure=("original_high_pressure", "sum"),
        adjusted_high_pressure=("adjusted_high_pressure", "sum"),
        max_score_change=("score_difference", "max")
    )
    .reset_index()
    .sort_values("changed_deliveries", ascending=False)
)

changed_match_audit = changed_match_audit.merge(
    matches[
        [
            "match_id",
            "match_date",
            "result",
            "first_inn_revised_overs",
            "second_inn_revised_overs"
        ]
    ],
    on="match_id",
    how="left"
)

changed_match_audit = changed_match_audit.merge(
    competitions[["comp_id", "match_format"]],
    left_on="match_id",
    right_on="match_id",
    how="left"
) if "comp_id" in matches.columns else changed_match_audit

print("Total changed deliveries:", len(changed))
print("Matches affected:", changed["match_id"].nunique())

print("\nLargest match-level impacts:")
display(changed_match_audit.head(15))

KeyError: 'match_id'

In [19]:
# Identify matches responsible for pressure classification changes

changed = eligible_test[
    eligible_test["original_high_pressure"]
    != eligible_test["adjusted_high_pressure"]
].copy()

changed_match_audit = (
    changed.groupby("match_id")
    .agg(
        changed_deliveries=("match_id", "size"),
        original_high_pressure=("original_high_pressure", "sum"),
        adjusted_high_pressure=("adjusted_high_pressure", "sum"),
        max_score_change=("score_difference", "max")
    )
    .reset_index()
    .sort_values("changed_deliveries", ascending=False)
)

# Attach match details, including competition ID
match_details = matches[
    [
        "match_id",
        "comp_id",
        "match_date",
        "result",
        "first_inn_revised_overs",
        "second_inn_revised_overs"
    ]
].copy()

changed_match_audit = changed_match_audit.merge(
    match_details,
    on="match_id",
    how="left"
)

# Attach match format using comp_id
changed_match_audit = changed_match_audit.merge(
    competitions[["comp_id", "match_format"]],
    on="comp_id",
    how="left"
)

print("=" * 75)
print("CHANGED DELIVERY — MATCH-LEVEL AUDIT")
print("=" * 75)

print("Total changed deliveries:", len(changed))
print("Matches affected:", changed["match_id"].nunique())

print("\nLargest match-level impacts:")
display(changed_match_audit.head(15))

CHANGED DELIVERY — MATCH-LEVEL AUDIT
Total changed deliveries: 550
Matches affected: 28

Largest match-level impacts:


,match_id,changed_deliveries,original_high_pressure,adjusted_high_pressure,max_score_change,comp_id,match_date,result,first_inn_revised_overs,second_inn_revised_overs,match_format
0,50137,44,0,44,26.590909,3298,2026-04-07,Rajasthan Royals Won by 27 runs,66.0,66.0,T20
1,41851,37,0,37,29.500000,2906,2026-02-01,West Indies Won by 6 runs(DLS Method),60.0,60.0,T20I
2,50190,36,0,36,11.166667,3433,2026-05-02,New Zealand Won by 6 wickets (DLS Method),90.0,90.0,T20I
3,17395,34,0,34,35.250000,1584,2022-09-23,India Won by 6 Wickets,48.0,48.0,T20I
4,37354,33,0,33,38.071429,2387,2024-11-14,Australia Won by 29 runs,42.0,42.0,T20I
5,40727,28,0,28,14.357143,2725,2025-04-18,Punjab Kings Won by 5 wickets,84.0,84.0,T20
6,48610,27,0,27,12.666667,3334,2026-01-30,England Won by 11 runs (DLS method),102.0,90.0,T20I
7,16753,25,0,25,24.136364,1543,2022-08-15,Afghanistan Won by 27 Runs,66.0,NaN,T20I
8,36208,23,0,23,43.500000,2628,2025-09-10,South Africa Won by 14 runs(DLS method),54.0,30.0,T20I
9,36098,22,0,22,20.192308,2453,2024-08-28,West Indies Won by 8 wickets (DLS),78.0,78.0,T20I


In [21]:
# Step 6G: Break down changed deliveries

# The revised-duration columns already exist in changed.
# Only attach comp_id and result from matches.
changed_detail = changed.merge(
    matches[["match_id", "comp_id", "result"]],
    on="match_id",
    how="left"
)

changed_detail = changed_detail.merge(
    competitions[["comp_id", "match_format"]],
    on="comp_id",
    how="left"
)

# Identify the applicable innings-specific revised limit
changed_detail["innings_limit_used"] = (
    changed_detail["first_inn_revised_overs"]
    .where(
        changed_detail["innings"] == 1,
        changed_detail["second_inn_revised_overs"]
    )
)

print("=" * 75)
print("CHANGED DELIVERIES — FORMAT BREAKDOWN")
print("=" * 75)

display(
    changed_detail.groupby("match_format")
    .agg(
        changed_deliveries=("match_id", "size"),
        affected_matches=("match_id", "nunique")
    )
    .reset_index()
)

print("\nBreakdown by innings:")
display(
    changed_detail.groupby(["match_format", "innings"])
    .agg(
        changed_deliveries=("match_id", "size"),
        affected_matches=("match_id", "nunique"),
        revised_limits=("innings_limit_used", "nunique")
    )
    .reset_index()
)

print("\nBreakdown by revised innings duration:")
display(
    changed_detail.groupby("innings_limit_used")
    .agg(
        changed_deliveries=("match_id", "size"),
        affected_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values("innings_limit_used")
)

print("\nChanged deliveries without a recorded innings limit:",
      changed_detail["innings_limit_used"].isna().sum())

CHANGED DELIVERIES — FORMAT BREAKDOWN


,match_format,changed_deliveries,affected_matches
0,T20,97,5
1,T20I,453,23



Breakdown by innings:


,match_format,innings,changed_deliveries,affected_matches,revised_limits
0,T20,1,36,3,3
1,T20,2,61,5,4
2,T20I,1,253,17,12
3,T20I,2,200,15,8



Breakdown by revised innings duration:


,innings_limit_used,changed_deliveries,affected_matches
0,30.0,32,2
1,42.0,43,2
2,48.0,60,3
3,54.0,46,3
4,60.0,59,2
5,66.0,85,3
6,72.0,21,1
7,78.0,22,1
8,84.0,43,2
9,90.0,91,5



Changed deliveries without a recorded innings limit: 0


In [26]:
# Step 6I: No Result exclusion sensitivity test

# 1. Identify No Result matches
no_result_ids = set(
    matches.loc[
        matches["result"].str.contains(
            "No Result", case=False, na=False
        ),
        "match_id"
    ]
)

print("No Result matches identified:", len(no_result_ids))

# 2. Exclude the same matches from both models
original_hp_no_nr = original_hp[
    ~original_hp["match_id"].isin(no_result_ids)
].copy()

adjusted_hp_no_nr = adjusted_hp[
    ~adjusted_hp["match_id"].isin(no_result_ids)
].copy()

# 3. Recalculate leaderboards
baseline_batters_no_nr = calculate_batter_ranking(
    original_hp_no_nr
)

adjusted_batters_no_nr = calculate_batter_ranking(
    adjusted_hp_no_nr
)

baseline_bowlers_no_nr = calculate_bowler_ranking(
    original_hp_no_nr
)

adjusted_bowlers_no_nr = calculate_bowler_ranking(
    adjusted_hp_no_nr
)

# 4. Compare Top 10 membership
batter_baseline_set = set(
    baseline_batters_no_nr.head(10)["striker_name"]
)
batter_adjusted_set = set(
    adjusted_batters_no_nr.head(10)["striker_name"]
)

bowler_baseline_set = set(
    baseline_bowlers_no_nr.head(10)["bowler_name"]
)
bowler_adjusted_set = set(
    adjusted_bowlers_no_nr.head(10)["bowler_name"]
)

# 5. Print results
print("\n" + "=" * 75)
print("NO RESULT EXCLUSION — LEADERBOARD SENSITIVITY")
print("=" * 75)

print("No Result matches excluded:", len(no_result_ids))
print("Original high-pressure deliveries:", len(original_hp_no_nr))
print("Adjusted high-pressure deliveries:", len(adjusted_hp_no_nr))

print("\nBATTERS")
print("Top 10 overlap:",
      len(batter_baseline_set & batter_adjusted_set), "/ 10")
print("Dropped:", sorted(batter_baseline_set - batter_adjusted_set))
print("New entries:", sorted(batter_adjusted_set - batter_baseline_set))

print("\nAdjusted batter Top 10:")
display(
    adjusted_batters_no_nr.head(10)[
        ["striker_name", "batter_score",
         "high_pressure_runs", "balls_faced"]
    ]
)

print("\nBOWLERS")
print("Top 10 overlap:",
      len(bowler_baseline_set & bowler_adjusted_set), "/ 10")
print("Dropped:", sorted(bowler_baseline_set - bowler_adjusted_set))
print("New entries:", sorted(bowler_adjusted_set - bowler_baseline_set))

print("\nAdjusted bowler Top 10:")
display(
    adjusted_bowlers_no_nr.head(10)[
        ["bowler_name", "bowler_score",
         "high_pressure_wickets", "legal_balls"]
    ]
)

No Result matches identified: 21

NO RESULT EXCLUSION — LEADERBOARD SENSITIVITY
No Result matches excluded: 21
Original high-pressure deliveries: 61946
Adjusted high-pressure deliveries: 62420

BATTERS
Top 10 overlap: 10 / 10
Dropped: []
New entries: []

Adjusted batter Top 10:


,striker_name,batter_score,high_pressure_runs,balls_faced
0,Suryakumar Yadav,97.684332,1116,575
1,Tim David,95.829493,1447,814
2,Nicholas Pooran,95.368664,1149,652
3,Andre Russell,94.746544,1035,600
4,Shreyas Iyer,93.813364,851,451
5,Heinrich Klaasen,93.387097,1064,580
6,Jos Buttler,92.937788,698,379
7,Marcus Stoinis,92.350230,950,556
8,Ashutosh Sharma,91.947005,481,267
9,Harry Brook,91.820276,562,303



BOWLERS
Top 10 overlap: 9 / 10
Dropped: ['Matheesha Pathirana']
New entries: ['Abbas Afridi']

Adjusted bowler Top 10:


,bowler_name,bowler_score,high_pressure_wickets,legal_balls
0,Haris Rauf,93.789398,40,402
1,Mitchell Starc,93.574499,28,237
2,Kuldeep Yadav,92.865330,55,627
3,Yuzvendra Chahal,92.693410,61,712
4,Shaheen Afridi,92.371060,34,358
5,Wanindu Hasaranga,92.048711,53,617
6,Mark Adair,91.554441,31,328
7,Andre Russell,91.038682,31,331
8,Josh Hazlewood,90.974212,41,481
9,Abbas Afridi,90.436963,19,150


In [25]:
# Step 6H: Audit outcomes of affected matches

outcome_audit = changed_detail.merge(
    matches[["match_id", "match_date"]],
    on="match_id",
    how="left"
)

outcome_audit["result_category"] = np.select(
    [
        outcome_audit["result"].str.contains(
            "DLS|D/L", case=False, na=False
        ),
        outcome_audit["result"].str.contains(
            "No Result", case=False, na=False
        ),
        outcome_audit["result"].str.contains(
            "Abandoned", case=False, na=False
        )
    ],
    [
        "DLS / D-L result",
        "No Result",
        "Abandoned"
    ],
    default="Other result"
)

outcome_summary = (
    outcome_audit.groupby("result_category")
    .agg(
        affected_matches=("match_id", "nunique"),
        changed_deliveries=("match_id", "size")
    )
    .reset_index()
    .sort_values("changed_deliveries", ascending=False)
)

print("=" * 75)
print("AFFECTED MATCHES — OUTCOME AUDIT")
print("=" * 75)

display(outcome_summary)

print("\nDetailed affected matches:")

display(
    outcome_audit[
        [
            "match_id",
            "match_date",
            "match_format",
            "innings",
            "innings_limit_used",
            "result",
            "result_category"
        ]
    ]
    .drop_duplicates()
    .sort_values(["match_date", "match_id"])
)

AFFECTED MATCHES — OUTCOME AUDIT


,result_category,affected_matches,changed_deliveries
0,DLS / D-L result,14,243
2,Other result,9,231
1,No Result,5,76



Detailed affected matches:


,match_id,match_date,match_format,innings,innings_limit_used,result,result_category
0,13612,2022-02-11,T20I,2,114.0,Australia Won by 20 runs (D/L method),DLS / D-L result
3,15546,2022-07-02,T20I,1,84.0,No Result,No Result
18,16753,2022-08-15,T20I,1,66.0,Afghanistan Won by 27 Runs,Other result
43,16754,2022-08-17,T20I,2,42.0,Ireland Won by 7 Wickets(D/L Method),DLS / D-L result
53,17395,2022-09-23,T20I,1,48.0,India Won by 6 Wickets,Other result
71,17395,2022-09-23,T20I,2,48.0,India Won by 6 Wickets,Other result
87,17765,2022-10-14,T20I,1,72.0,No Result,No Result
108,19959,2023-03-25,T20I,1,66.0,West Indies won by 3 wickets,Other result
124,21070,2023-03-27,T20I,2,48.0,Bangladesh Won by 22 Runs (DLS method),DLS / D-L result
145,21071,2023-03-29,T20I,1,102.0,Bangladesh Won by 77 Runs,Other result


In [27]:

# Step 6J: Compare all four leaderboard scenarios

def top10_names(df, name_column):
    return list(df.head(10)[name_column])

# Original saved Top 10
saved_batters = set(
    batter.loc[batter["final_rank"] <= 10, "striker_name"]
)
saved_bowlers = set(
    bowler.loc[bowler["final_rank"] <= 10, "bowler_name"]
)

# Recalculated scenarios
scenarios = {
    "Original full dataset": (
        baseline_batters,
        baseline_bowlers
    ),
    "Original excluding No Result": (
        baseline_batters_no_nr,
        baseline_bowlers_no_nr
    ),
    "Adjusted full dataset": (
        adjusted_batters,
        adjusted_bowlers
    ),
    "Adjusted excluding No Result": (
        adjusted_batters_no_nr,
        adjusted_bowlers_no_nr
    )
}

print("=" * 75)
print("FOUR-SCENARIO LEADERBOARD COMPARISON")
print("=" * 75)

for scenario, (bat_df, bowl_df) in scenarios.items():

    bat_names = set(top10_names(bat_df, "striker_name"))
    bowl_names = set(top10_names(bowl_df, "bowler_name"))

    print(f"\n{scenario}")
    print("-" * 55)

    print(
        "Batter overlap with saved original:",
        len(bat_names & saved_batters), "/ 10"
    )
    print(
        "Batter differences:",
        sorted(saved_batters - bat_names),
        "| New:",
        sorted(bat_names - saved_batters)
    )

    print(
        "Bowler overlap with saved original:",
        len(bowl_names & saved_bowlers), "/ 10"
    )
    print(
        "Bowler differences:",
        sorted(saved_bowlers - bowl_names),
        "| New:",
        sorted(bowl_names - saved_bowlers)
    )

FOUR-SCENARIO LEADERBOARD COMPARISON

Original full dataset
-------------------------------------------------------
Batter overlap with saved original: 10 / 10
Batter differences: [] | New: []
Bowler overlap with saved original: 10 / 10
Bowler differences: [] | New: []

Original excluding No Result
-------------------------------------------------------
Batter overlap with saved original: 10 / 10
Batter differences: [] | New: []
Bowler overlap with saved original: 10 / 10
Bowler differences: [] | New: []

Adjusted full dataset
-------------------------------------------------------
Batter overlap with saved original: 9 / 10
Batter differences: ['Harry Brook'] | New: ['Naman Dhir']
Bowler overlap with saved original: 9 / 10
Bowler differences: ['Matheesha Pathirana'] | New: ['Adam Zampa']

Adjusted excluding No Result
-------------------------------------------------------
Batter overlap with saved original: 10 / 10
Batter differences: [] | New: []
Bowler overlap with saved original: 9 

In [28]:
print("=" * 75)
print("AFFECTED PLAYER COMPARISON")
print("=" * 75)

print("\nBATTERS — Original vs adjusted, excluding No Result")

batter_names = [
    "Harry Brook",
    "Naman Dhir"
]

for name in batter_names:
    print(f"\n{name}")

    print("Original:")
    display(
        baseline_batters_no_nr[
            baseline_batters_no_nr["striker_name"] == name
        ]
    )

    print("Adjusted:")
    display(
        adjusted_batters_no_nr[
            adjusted_batters_no_nr["striker_name"] == name
        ]
    )


print("\nBOWLERS — Original vs adjusted, excluding No Result")

bowler_names = [
    "Matheesha Pathirana",
    "Abbas Afridi",
    "Adam Zampa"
]

for name in bowler_names:
    print(f"\n{name}")

    print("Original:")
    display(
        baseline_bowlers_no_nr[
            baseline_bowlers_no_nr["bowler_name"] == name
        ]
    )

    print("Adjusted:")
    display(
        adjusted_bowlers_no_nr[
            adjusted_bowlers_no_nr["bowler_name"] == name
        ]
    )

AFFECTED PLAYER COMPARISON

BATTERS — Original vs adjusted, excluding No Result

Harry Brook
Original:


,striker_id,striker_name,high_pressure_runs,balls_faced,fours,sixes,pressure_deliveries,matches_played,strike_rate,boundary_runs,boundary_percentage,runs_percentile,strike_rate_percentile,boundary_percentile,sample_reliability,batter_score
9,6280,Harry Brook,554,294,51,28,315,24,188.435374,372,67.148014,0.909931,0.95612,0.7806,1.0,91.789838


Adjusted:


,striker_id,striker_name,high_pressure_runs,balls_faced,fours,sixes,pressure_deliveries,matches_played,strike_rate,boundary_runs,boundary_percentage,runs_percentile,strike_rate_percentile,boundary_percentile,sample_reliability,batter_score
9,6280,Harry Brook,562,303,52,28,324,24,185.478548,376,66.903915,0.914747,0.951613,0.778802,1.0,91.820276



Naman Dhir
Original:


,striker_id,striker_name,high_pressure_runs,balls_faced,fours,sixes,pressure_deliveries,matches_played,strike_rate,boundary_runs,boundary_percentage,runs_percentile,strike_rate_percentile,boundary_percentile,sample_reliability,batter_score
18,37052,Naman Dhir,411,229,32,26,236,22,179.475983,284,69.099757,0.856813,0.926097,0.852194,1.0,89.838337


... (10 outputs left out)


Adam Zampa
Original:


,bowler_id,bowler_name,high_pressure_wickets,legal_balls,pressure_deliveries,matches_played,wickets_per_24_balls,wickets_percentile,wicket_rate_percentile,sample_sufficiency,bowler_score
10,2636,Adam Zampa,37,438,446,43,2.027397,0.944126,0.756447,1.0,90.179083


Adjusted:


,bowler_id,bowler_name,high_pressure_wickets,legal_balls,pressure_deliveries,matches_played,wickets_per_24_balls,wickets_percentile,wicket_rate_percentile,sample_sufficiency,bowler_score
10,2636,Adam Zampa,38,444,452,43,2.054054,0.945559,0.762178,1.0,90.415473
